# CPU-only environment probe

Requests the exact image digest the 2026-10-03 verification run used:

    gcr.io/kaggle-private-byod/python@sha256:37c64f7dd9c54116ecd1bcc88817c5469b88387388fade02bfa8bf3fc647d461

Reports the interpreter and wheel-tag compatibility, then runs the comparison's own wheelhouse
install and verifies required imports and the compiler version plus per-file hashes.

**No GPU, model startup, agent evaluation or submission.** A pass establishes environment
compatibility only: it says nothing about vLLM operation on GPU or about candidate performance.
The four-run comparison and the frozen candidates are untouched by this probe.


In [ ]:
# ============================ CONFIG ============================
EXPECTED_IMAGE = "gcr.io/kaggle-private-byod/python@sha256:37c64f7dd9c54116ecd1bcc88817c5469b88387388fade02bfa8bf3fc647d461"
EXPECTED_PYTHON = (3, 12)      # observed on the successful run; verified below, never assumed
from pathlib import Path
WORKING_DIR = Path("/kaggle/working")
RESULTS = WORKING_DIR / "pilot"; RESULTS.mkdir(parents=True, exist_ok=True)
print("probe: CPU only, no GPU or model startup")


In [ ]:
# Step 2 and 3: interpreter, platform, supported tags, and compatibility of EVERY wheelhouse file.
# Written to disk BEFORE the install so it survives an install failure.
import json, platform, sys, sysconfig
from pathlib import Path

probe = {
    "sys_version": sys.version,
    "sys_version_info": list(sys.version_info[:3]),
    "sys_executable": sys.executable,
    "platform": platform.platform(),
    "machine": platform.machine(),
    "python_implementation": platform.python_implementation(),
    "sysconfig_platform": sysconfig.get_platform(),
    "expected_image": EXPECTED_IMAGE,
    "expected_python": list(EXPECTED_PYTHON),
}
print("python     :", probe["sys_version_info"], probe["sys_executable"])
print("platform   :", probe["platform"], "|", probe["sysconfig_platform"])
probe["image_python_as_expected"] = tuple(sys.version_info[:2]) == EXPECTED_PYTHON
print("python matches the recorded successful run:", probe["image_python_as_expected"])

from packaging.tags import sys_tags
from packaging.utils import parse_wheel_filename
supported = [str(t) for t in sys_tags()]
probe["supported_tag_count"] = len(supported)
probe["supported_tags_sample"] = supported[:25]
print("supported tags:", len(supported), "e.g.", supported[:5])

WH = Path("/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse")
probe["wheelhouse_dir"] = str(WH)
probe["wheelhouse_present"] = WH.is_dir()
files = sorted(p.name for p in WH.iterdir() if p.is_file()) if WH.is_dir() else []
wheels = [f for f in files if f.endswith(".whl")]
probe["file_count"] = len(files)
probe["wheel_count"] = len(wheels)
probe["non_wheel_files"] = [f for f in files if not f.endswith(".whl")]

sup = set(supported)
compatible, incompatible = [], []
for n in wheels:
    try:
        _, _, _, tags = parse_wheel_filename(n)
    except Exception as exc:
        incompatible.append({"wheel": n, "reason": f"unparsable: {exc}", "wheel_tags": []})
        continue
    tl = sorted(str(t) for t in tags)
    if any(t in sup for t in tl):
        compatible.append({"wheel": n})
    else:
        incompatible.append({"wheel": n, "reason": "no tag matches this interpreter",
                             "wheel_tags": tl[:6]})
probe["compatible_count"] = len(compatible)
probe["incompatible"] = incompatible
probe["all_wheels_compatible"] = (len(wheels) > 0 and not incompatible)

print(f"wheels: {len(wheels)} total, {len(compatible)} compatible, {len(incompatible)} incompatible")
for w in incompatible:
    print("   INCOMPATIBLE:", w["wheel"], "|", w.get("reason"), "|", w.get("wheel_tags"))

(RESULTS / "env_probe.json").write_text(json.dumps(probe, indent=2), encoding="utf-8")
print("saved", RESULTS / "env_probe.json")
PROBE = probe


In [ ]:
# Step 4 gate: only attempt the install when every wheel is compatible. No wheel is ever skipped.
if not PROBE["wheelhouse_present"]:
    raise SystemExit("wheelhouse dataset not mounted; diagnostics saved")
if not PROBE["all_wheels_compatible"]:
    raise SystemExit(f"{len(PROBE['incompatible'])} incompatible wheel(s) for this interpreter; "
                     "diagnostics saved, install not attempted, nothing substituted or skipped")
print("all wheelhouse wheels are tag-compatible; running the existing install")


In [ ]:
# Wheelhouse install. Exit status is checked, not piped away.
import glob, importlib, os, subprocess, sys
from pathlib import Path
os.environ.update({
    "LITELLM_LOCAL_MODEL_COST_MAP": "True", "TRANSFORMERS_NO_TF": "1",
    "VLLM_WORKER_MULTIPROC_METHOD": "spawn", "VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS": "1",
    "VLLM_ENGINE_READY_TIMEOUT_S": "1200", "VLLM_NO_USAGE_STATS": "1",
    "OTEL_SDK_DISABLED": "true", "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
})
WHEELHOUSE_DIR = Path("/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse")
for pat in ("/usr/local/lib/python*/dist-packages/*cutlass*.pth",
            "/usr/local/lib/python*/site-packages/*cutlass*.pth"):
    for pth in glob.glob(pat):
        try:
            os.unlink(pth)
        except OSError:
            pass
tmp_whl = Path("/tmp/wheelhouse"); tmp_whl.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE_DIR.glob("*.whl"):
    if "cutlass" in w.name.lower():
        continue
    name = w.name.replace("cu128", "+cu128") if ("cu128" in w.name and "+" not in w.name) else w.name
    if not (tmp_whl / name).exists():
        os.symlink(w, tmp_whl / name)
wheels = sorted(str(w) for w in tmp_whl.glob("*.whl"))
print("installing", len(wheels), "wheels...")
proc = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps",
                       "--force-reinstall", *wheels], capture_output=True, text=True)
if proc.returncode != 0:
    print(proc.stdout[-2000:]); print(proc.stderr[-2000:])
    raise SystemExit(f"wheelhouse install failed rc={proc.returncode}")
importlib.invalidate_caches()
import importlib.metadata as md
VERSIONS = {}
for p in ("swegemma", "adk-submission", "adk-eval-core", "google-adk", "vllm", "transformers", "litellm"):
    try:
        VERSIONS[p] = md.version(p)
    except Exception as e:
        VERSIONS[p] = "MISSING: " + repr(e)[:60]
print(json.dumps(VERSIONS, indent=2))


In [ ]:
# Compiler-identity gate, lifted verbatim from the comparison notebook.
import hashlib, json
from pathlib import Path
EXPECTED_COMPILER_FILES = {'compiler.py': '39cc4884699faf147ca1b326664e888c1861f750fe36525b840b5f812d593eed', 'context.py': '3c77519415b1dd2cc97b42bcbe29d32dd85f8188ca7eba5b5fb8636bce92c908', 'discovery.py': 'ca19fc75fb3baa42e85438b41248a06c35987cbf26ef652159b302752d0f191c', 'errors.py': '7e121e7c6581288788500cd1a8deceb003a74ec150d5477c37f794971a5f4818', 'limits.py': 'afde5056f2df1fd77b53e0495712d03f60b3e246d4393617dd800d970f7482e2', 'paths.py': '7c4b7050141bdb05a9e6849be11f6440f2a18082a364623226c4343649b2f59f', 'registry.py': 'c17f9770c2f3663309dd61807d2ae4e7d75bbd06b7c674847f655d218e82cf7a', 'schema.py': '6554de60cfa208dcb0b095f4236271795b2955adf87f3be3c40dfa3d62521c5d', 'server.py': 'd60b3fe34dd7f6e359b003db82b68434757c74b1030862bacda5cff6164f91eb', 'yaml_loader.py': '01d644ef7f5acc216506245fd7e115e7096641e62e825c6873c83e148b23c69c', '__init__.py': 'ebe048f521fd5bcf46dad132f3e3dea2506df79bf471d560a70ce5b343ca387a', 'builders/llm.py': '97c27324ce60ac6dc466941bcc19f5482f7693af9fc02fb760a40526c84c10b5', 'builders/sub_agents.py': 'ba6162832b1ca4e023e2848c6391242c18cbe9dadcfd560a6ebef81108c35b5a', 'builders/workflows.py': 'd8d7ccf9a51d57daec58c7231e2a62621a3ecca269e71ee64c57062cadadf980', 'builders/__init__.py': 'ec6273c54cfb08f22777146c1636be4f13d4d00a67a17d9ae7a19e3a8f18f6c1', 'resolvers/callbacks.py': '332ce0732aa3eb1b555630c57683f51e59d5caee3f519371d376210e9ae33e29', 'resolvers/generation.py': 'ba8bf237261220d46843c8d71118faa387fc07f876f072756dff44d641dd1c14', 'resolvers/models.py': '39070f6ae52ad8880750b51af373da8762850d3a8ecc65c6f11744afe4cd5066', 'resolvers/tools.py': 'ae44a5ca815a05edfd763b92283d0444c77755afbd827fc35bcb87eb2633259e', 'resolvers/__init__.py': 'b88fe281c2b927a046e6485c7e0966b1808a5c8d0a0849cd85824ffb04ae53be'}
def verify_runtime_compiler(root=None, version=None):
    import importlib.metadata as metadata
    if root is None:
        import adk_submission
        root = Path(adk_submission.__file__).parent
    if version is None:
        version = metadata.version('adk-submission')
    found = {p.relative_to(root).as_posix(): hashlib.sha256(p.read_bytes()).hexdigest()
             for p in root.rglob('*.py')}
    record = {'version': version, 'python_files': found,
              'matches': version == '0.2.12' and found == EXPECTED_COMPILER_FILES}
    (RESULTS / 'compiler_identity.json').write_text(json.dumps(record, indent=2))
    assert record['matches'], 'Runtime compiler drift: refuse model startup'


In [ ]:
# Step 4 continued: required imports and compiler identity. Reuses the comparison's own gate.
import importlib, importlib.metadata as md, json, traceback
report = {"versions": VERSIONS, "imports": {}, "compiler": None}
REQUIRED = ["swegemma", "adk_submission", "adk_eval_core", "google.adk", "vllm",
            "transformers", "litellm", "packaging"]
for mod in REQUIRED:
    try:
        importlib.import_module(mod)
        report["imports"][mod] = "ok"
    except Exception as exc:
        report["imports"][mod] = f"FAILED: {type(exc).__name__}: {exc}"
print(json.dumps(report["imports"], indent=2))
(RESULTS / "probe_imports.json").write_text(json.dumps(report, indent=2), encoding="utf-8")

failed = {k: v for k, v in report["imports"].items() if v != "ok"}
if failed:
    raise SystemExit(f"required imports failed: {sorted(failed)}; diagnostics saved")

try:
    verify_runtime_compiler()
    report["compiler"] = "version and per-file hashes match 0.2.12"
    print("compiler identity: PASS (version 0.2.12 and per-file hashes)")
except AssertionError as exc:
    report["compiler"] = f"FAILED: {exc}"
    (RESULTS / "probe_imports.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
    raise
(RESULTS / "probe_imports.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
print("PROBE PASS: environment compatibility only. GPU/vLLM operation and candidate performance "
      "are NOT established by this probe.")
